<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# DXQuant · Derivatives Analytics with Python
## Notebook 6 · Heston Pricing: Fourier vs Monte Carlo

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/dxquant/blob/main/notebooks/06_dxquant_heston_fourier_and_monte_carlo.ipynb)


## Notebook Goals

This notebook illustrates two complementary pricing routes for the Heston (1993) model:

- Carr–Madan Fourier pricing based on the characteristic function, and
- Monte Carlo pricing based on simulated spot and variance paths.

The objective is to show consistency and typical numerical trade-offs, not to provide a
production calibration setup.

## Setup

In Google Colab, select **Runtime → Run all** with a standard CPU runtime. This cell installs DXQuant and Matplotlib if they are missing. No GPU, Google Drive mount, or API credentials are needed. The examples use synthetic inputs.

Before the first PyPI release, installation uses the public GitHub `main` branch. When running from a local checkout, the setup cell installs the local package instead.


In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path

requirements = []
if importlib.util.find_spec("dxquant") is None:
    package_source = (
        "dxquant @ git+https://github.com/yhilpisch/dxquant.git@main"
    )
    for directory in (Path.cwd(), Path.cwd().parent):
        local_package = directory
        if (local_package / "pyproject.toml").is_file():
            package_source = str(local_package.resolve())
            break
    requirements.append(package_source)
if importlib.util.find_spec("matplotlib") is None:
    requirements.append("matplotlib>=3.8")
if requirements:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *requirements]
    )


In [ ]:
import datetime as dt
import math

import numpy as np

import dxquant as dx

np.set_printoptions(precision=6, suppress=True)

ENABLE_PLOTS = False
plt = None
if ENABLE_PLOTS:
    try:
        import matplotlib.pyplot as plt
    except ModuleNotFoundError:
        plt = None

## 1. Model Setup

We use a single maturity and a strike grid. Discounting is kept external to highlight the
pricing engines.


In [ ]:
pricing_date = dt.date(2024, 1, 1)
maturity_date = dt.date(2025, 1, 1)

spot = 100.0
rate = 0.01

maturity = float(dx.year_fractions([pricing_date, maturity_date])[1])
discount = float(math.exp(-rate * maturity))

strikes = np.linspace(80.0, 120.0, 9)
strikes

## 2. Fourier Pricing (Carr–Madan)

The Fourier pricer integrates an expression built from the characteristic function of the
log-price.

In [ ]:
heston = dx.HestonProcess(
    kappa=2.0,
    theta=0.04,
    sigma=0.6,
    rho=-0.5,
    drift=rate,
)

variance_0 = heston.theta
phi = heston.characteristic_function(
    tau=maturity,
    spot=spot,
    variance=variance_0,
)

fourier = dx.FourierPricer(
    dampening=1.2,
    integration_limit=200.0,
    steps=10_000,
)

fourier_prices = np.array(
    [
        fourier.price_call(
            characteristic_fn=phi,
            spot=spot,
            strike=float(k),
            discount_factor=discount,
        )
        for k in strikes
    ],
    dtype=float,
)

fourier_prices

## 3. Monte Carlo Pricing (Shared Paths, Multiple Strikes)

For multi-strike comparisons it is convenient to generate a single simulated terminal spot
vector and evaluate all strike payoffs on the same draw.

In [ ]:
paths = 60_000
steps = 128
grid = dx.uniform_time_grid(maturity=maturity, num_steps=steps)

heston_mc = dx.HestonProcess(
    kappa=heston.kappa,
    theta=heston.theta,
    sigma=heston.sigma,
    rho=heston.rho,
    drift=heston.drift,
    seed=123,
)

spot_paths, var_paths = heston_mc.simulate_paths(
    spot=spot,
    variance=variance_0,
    time_grid=grid,
    paths=paths,
)

terminal = spot_paths[:, -1]
payoffs = np.maximum(terminal[:, None] - strikes[None, :], 0.0)
discounted = discount * payoffs
mc_prices = np.mean(discounted, axis=0)

mc_prices

## 4. Comparison

The two estimators will not match perfectly because they have different numerical errors.
Fourier integration has truncation and discretization error; Monte Carlo has sampling error.


In [ ]:
diff = mc_prices - fourier_prices
rel = diff / np.maximum(1e-12, fourier_prices)

for k, f, m, d, r in zip(
    strikes, fourier_prices, mc_prices, diff, rel, strict=True
):
    msg = (
        f"K={k:6.2f}  "
        f"Fourier={f:9.5f}  MC={m:9.5f}  "
        f"diff={d: .5f}  rel={r: .3%}"
    )
    print(msg)

if plt is not None:
    plt.figure(figsize=(9, 4))
    plt.plot(strikes, fourier_prices, marker="o", label="Fourier")
    plt.plot(strikes, mc_prices, marker="x", label="Monte Carlo")
    plt.title("Heston call prices by strike")
    plt.xlabel("strike")
    plt.ylabel("price")
    plt.legend()
    plt.show()

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">